In [3]:
# If you need to install packages uncomment and run
!pip install matplotlib pandas seaborn opencv-python-headless tensorflow pillow

import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from PIL import Image
import random
import pandas as pd

sns.set(style="whitegrid")


## Define paths and check counts

In [ ]:
from pathlib import Path

# auto-detect if inside notebooks/
if Path.cwd().name == "notebooks":
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd()

# Corrected path to match actual dataset location
DATA_DIR = PROJECT_DIR / "dataset" / "Classification Dataset"
train_dir = DATA_DIR / "train"
val_dir = DATA_DIR / "valid"
test_dir = DATA_DIR / "test"

print("PROJECT_DIR =", PROJECT_DIR)
print("DATA_DIR =", DATA_DIR)
print("Train exists:", train_dir.exists())
print("Valid exists:", val_dir.exists())
print("Test exists:", test_dir.exists())

# If dirs don't exist, list what's in dataset folder
if not train_dir.exists():
    dataset_folder = PROJECT_DIR / "dataset"
    if dataset_folder.exists():
        print("\nContents of dataset folder:")
        for item in dataset_folder.iterdir():
            print(f"  - {item.name}")


DATA_DIR = f:\Aerial Object Classification & Detection\data\classification_dataset
Train exists: False
Valid exists: False
Test exists: False


## Build DataFrame of file paths and labels

In [13]:
def paths_df(root):
    rows=[]
    for split in ["train","valid","test"]:
        split_dir = root / split
        if not split_dir.exists(): 
            print(f"Warning: {split_dir} does not exist")
            continue
        for cls in sorted(os.listdir(split_dir)):
            cls_dir = split_dir / cls
            if not cls_dir.is_dir(): continue
            for img in cls_dir.glob("*"):
                if img.suffix.lower() in [".jpg",".jpeg",".png"]:
                    rows.append({"split":split,"class":cls,"path":str(img)})
    return pd.DataFrame(rows)

df = paths_df(DATA_DIR)
if len(df) > 0:
    display(df.groupby(["split","class"]).size().unstack(fill_value=0))
else:
    print("No images found. Please check if the data directory structure is correct.")
    print("Expected structure: DATA_DIR/[train|valid|test]/[class_name]/*.jpg")


No images found. Please check if the data directory structure is correct.
Expected structure: DATA_DIR/[train|valid|test]/[class_name]/*.jpg


## Visualize random samples

In [ ]:
def plot_samples(n=6, split="train"):
    samples = df[df["split"]==split].sample(min(n,len(df[df["split"]==split])))
    fig, axes = plt.subplots(1, len(samples), figsize=(4*len(samples),4))
    if len(samples)==1:
        axes=[axes]
    for ax, (_, row) in zip(axes, samples.iterrows()):
        img = Image.open(row["path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{row['class']}\n{Path(row['path']).name}")
        ax.axis("off")
    plt.show()

plot_samples(n=6, split="train")


## Image size distribution

In [ ]:
def size_distribution(n=500):
    sizes=[]
    for _, row in df.sample(min(n,len(df))).iterrows():
        try:
            im = Image.open(row["path"])
            sizes.append(im.size)
        except:
            continue
    sizes = np.array(sizes)
    w,h = sizes[:,0], sizes[:,1]
    print("Avg size (w,h):", np.mean(w).round(1), np.mean(h).round(1))
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=w, y=h)
    plt.xlabel("Width"); plt.ylabel("Height"); plt.title("Image size distribution (sample)")
    plt.show()

size_distribution()
